In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:

%cd /content/drive/MyDrive/Manuscript_Segmentation/ManuscriptSegmentation
%pip install -r requirements.txt
!pip install aspose-words

/content/drive/MyDrive/Manuscript_Segmentation/ManuscriptSegmentation


In [17]:
import unicodedata
import pandas as pd
import numpy as np

import os
print(os.getcwd())  # Покажет текущую папку
print(os.listdir()) # П

/content/drive/MyDrive/Manuscript_Segmentation/ManuscriptSegmentation
['requirements.txt', 'notebooks', '.ipynb_checkpoints', 'data']


In [18]:
# 1. Установка системной утилиты
!apt-get install -y antiword

import os
import re
import unicodedata
import notebooks.get_words_corpus as wc

text = wc.get_raw_lines_from_doc('data/193-210.doc')
words_corpus = wc.get_clean_words(text)

print(f"Корпус собран. Всего слов: {len(words_corpus)}")
print("Первые 15 слов корпуса:")
print(words_corpus[:15])
#-----Пока не обрабатывает года, но обрабатывает номера страницу, буквы между рукописями

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
antiword is already the newest version (0.37-16).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
безазорнъ҇хъ лиць кромѣ ходѧть · женѹ имать или рабъ҇нѧ · безазорьноѥ себе ѡ семь хранѧ · аще ли пре
Корпус собран. Всего слов: 4687
Первые 15 слов корпуса:
['безазорнъ҇хъ', 'лиць', 'кромѣ', 'ходѧть', 'женѹ', 'имать', 'или', 'рабъ҇нѧ', 'безазорьноѥ', 'себе', 'ѡ', 'семь', 'хранѧ', 'аще', 'ли']


In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split

split_index = int(len(words_corpus) * 0.7)
train_text = words_corpus[:split_index]
test_text = words_corpus[split_index:]

In [20]:
def prepare_chars_and_labels(words_list):
    chars = []
    labels = []
    for word in words_list:
        if not word or not isinstance(word, str):
            continue
        # Первая буква слова — класс 1 (Begin)
        chars.append(word[0].lower())
        labels.append(1)
        # Остальные буквы слова — класс 0 (Inside)
        for char in word[1:]:
            chars.append(char.lower())
            labels.append(0)
    return chars, labels

train_chars, train_labels = prepare_chars_and_labels(train_text)
test_chars, test_labels = prepare_chars_and_labels(test_text)

print("склеенный текст:", train_chars[:15])
print("метки:          ", train_labels[:15])

склеенный текст: ['б', 'е', 'з', 'а', 'з', 'о', 'р', 'н', 'ъ', '҇', 'х', 'ъ', 'л', 'и', 'ц']
метки:           [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]


In [21]:
unique_chars = sorted(list(set(train_chars + test_chars)))
char_to_idx = {char: idx + 1 for idx, char in enumerate(unique_chars)}
char_to_idx['[PAD]'] = 0  # Токен отступа для выравнивания длин

# Переводим буквы в их числовые индексы
train_ids = [char_to_idx[c] for c in train_chars]
test_ids = [char_to_idx[c] for c in test_chars]

print("Размер старославянского словаря из рукописей:", len(unique_chars))


Размер старославянского словаря из рукописей: 43


In [50]:
class BiLSTMSegmentation(nn.Module):
  def __init__(self, output_dim, bidirectional, vocab_size, embedding_dim = 64, hidden_dim = 64):
      super().__init__()
      self.embedding = nn.Embedding(vocab_size + 1, embedding_dim)
      self.fc = nn.Linear(hidden_dim * 2, output_dim)
      self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=bidirectional, batch_first=True)

  def forward(self, x):
    embedding = self.embedding(x)
    out, hidden_out = self.lstm(embedding)
    out = self.fc(out)
    return out


In [29]:
class CharSegmentationDataset(Dataset):
    def __init__(self, ids_list, labels_list, seq_len=64):
        self.seq_len = seq_len
        # Нарезаем на куски фиксированной длины
        num_samples = len(ids_list) // seq_len

        self.X = torch.tensor(ids_list[:num_samples * seq_len]).view(num_samples, seq_len)
        self.y = torch.tensor(labels_list[:num_samples * seq_len]).view(num_samples, seq_len)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CharSegmentationDataset(train_ids, train_labels, seq_len=64)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = CharSegmentationDataset(test_ids, test_labels, seq_len=64)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [54]:
criterion = nn.CrossEntropyLoss()
model = BiLSTMSegmentation(2, True, len(unique_chars))
optimizer = optim.Adam(model.parameters(), lr = 0.01)

for i in range(10):
  total_loss = 0
  for batch in train_loader:
    X, y = batch
    optimizer.zero_grad()
    pred = model(X)
    pred = pred[:, 0, :]
    y = y[:, 0]
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epoch {i + 1}, Loss: {total_loss / len(train_loader)}")

   # Эпохи

Epoch 1, Loss: 0.5256060719490051
Epoch 2, Loss: 0.3733265340328217
Epoch 3, Loss: 0.32762927412986753
Epoch 4, Loss: 0.31352382302284243
Epoch 5, Loss: 0.27313581109046936
Epoch 6, Loss: 0.21523061096668245
Epoch 7, Loss: 0.14465608447790146
Epoch 8, Loss: 0.1088394582271576
Epoch 9, Loss: 0.061733578145503995
Epoch 10, Loss: 0.03610365390777588


In [1]:
def segment_raw_text(input_string, model, char_to_idx):
    model.eval()
    with torch.no_grad():

        ids = [char_to_idx.get(char.lower(), 0) for char in input_string]
        input_tensor = torch.tensor([ids])

        logits = model(input_tensor)

        predictions = torch.argmax(logits, dim=-1).squeeze(0).tolist()
        и
        result = []
        for char, pred in zip(input_string, predictions):
            if pred == 1 and result:
                result.append(" ")
            result.append(char)

        return "".join(result)

# Проверка
raw_test_string = "внесеныисправленияприподведенииразночтений"
segmented_output = segment_raw_text(raw_test_string, model, char_to_idx)
segmented_output

NameError: name 'model' is not defined